# Pass 3B — Corrected Leakage-Controlled Split + VGG16 Baseline

This notebook corrects the previous Pass 3B implementation.

**Key correction:** the uploaded Pass 3A CSV is an audit manifest; it does not already contain a train/validation/test split. This notebook therefore **creates the locked split first**, verifies it, saves it, and only then uses it for the VGG16 experiments.

## Design

- Binary task: **benign vs malignant**
- `normal` images remain in the audit manifest but are excluded from the lesion-classification experiment.
- Exact duplicate images are grouped by MD5 and cannot cross partitions.
- `filename_number` is retained for audit purposes but is **not treated as a patient identifier**.
- Split proportions are generated from the actual data; no assumed counts are hard-coded.
- A single locked split is reused for whole-image, raw-ROI, and preprocessed-ROI experiments.


In [ ]:
from pathlib import Path
import os, random, json, hashlib, re, warnings

import numpy as np
import pandas as pd

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------------
# INPUT: uploaded Pass 3A audit manifest
# ------------------------------------------------------------------
AUDIT_CSV = Path("/mnt/data/47fa9937-ba38-44b4-8b7f-1f7f36ee3219.csv")

# ------------------------------------------------------------------
# EDIT these roots for your local project
# ------------------------------------------------------------------
DATA_ROOT = Path("../data/Dataset_BUSI_with_GT")
RAW_ROI_ROOT = Path("../data/raw_roi_dataset")
PREP_ROI_ROOT = Path("../data/preprocessed_roi_dataset")

OUTPUT_DIR = Path("../results/pass3b_vgg16")
SPLIT_DIR = OUTPUT_DIR / "split"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
FIG_DIR = OUTPUT_DIR / "figures"

for d in [OUTPUT_DIR, SPLIT_DIR, MODEL_DIR, PRED_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Audit CSV:", AUDIT_CSV)
print("Exists:", AUDIT_CSV.exists())


In [ ]:
# Load and inspect the Pass 3A audit manifest
if not AUDIT_CSV.exists():
    raise FileNotFoundError(f"Cannot find audit CSV: {AUDIT_CSV}")

audit = pd.read_csv(AUDIT_CSV)

print("Shape:", audit.shape)
print("Columns:")
print(audit.columns.tolist())

required = {"image_path", "filename", "class", "md5"}
missing = required - set(audit.columns)
if missing:
    raise ValueError(f"Required audit columns missing: {sorted(missing)}")

print("\nClass distribution:")
print(audit["class"].value_counts())


## 1. Establish the binary classification cohort

The BUSI audit contains benign, malignant, and normal images.

For the existing VGG16 lesion-classification pipeline, the experimental cohort is **benign + malignant**. Normal images are not silently relabeled as benign or used in the binary classification task.

This distinction is important because the ROI pipeline is lesion-focused.


In [ ]:
df = audit[audit["class"].isin(["benign", "malignant"])].copy()

label_map = {"benign": 0, "malignant": 1}
df["label"] = df["class"].map(label_map)

print("Binary cohort:", len(df))
print(df["class"].value_counts())

assert len(df) == 647, f"Expected 647 benign/malignant images from the audit, got {len(df)}"


## 2. Define the leakage-control grouping

The Pass 3A audit showed that the filename number repeats across diagnostic classes, so it is **not a defensible patient identifier**.

The MD5 hash identifies exact duplicate files. All rows sharing the same MD5 are therefore forced into the same split.

For ordinary unique images, the MD5 itself is the split group.

This controls **exact-image duplication leakage**. It does not establish patient-level independence.


In [ ]:
df["split_group_id"] = df["md5"].astype(str)

group_sizes = df.groupby("split_group_id").size()

print("Images:", len(df))
print("Unique exact-image groups:", group_sizes.size)
print("Groups containing >1 image:", int((group_sizes > 1).sum()))
print("Maximum group size:", int(group_sizes.max()))

duplicate_groups = (
    df[df["split_group_id"].isin(group_sizes[group_sizes > 1].index)]
    .sort_values(["split_group_id", "filename"])
)

if len(duplicate_groups):
    print("\nExact duplicate groups:")
    display(duplicate_groups[["filename", "class", "md5", "split_group_id"]])


## 3. Create the locked stratified group split

Target proportions:

- 70% training
- 15% validation
- 15% test

Because exact duplicate groups must remain intact, the final counts may differ slightly from exactly 70/15/15.

The allocation is performed at the **group level**, while preserving class composition as closely as possible.

No previously assumed split counts are used.


In [ ]:
# Group-level class representation.
# For this dataset, the duplicate group identified in the audit contains
# one benign and one malignant image, so its class contribution is tracked
# explicitly.

group_table = (
    df.groupby("split_group_id")
      .agg(
          n_images=("filename", "size"),
          n_benign=("label", lambda x: int((x == 0).sum())),
          n_malignant=("label", lambda x: int((x == 1).sum())),
      )
      .reset_index()
)

rng = np.random.default_rng(SEED)

# Separate multi-image groups from singleton groups.
multi = group_table[group_table["n_images"] > 1].copy()
single = group_table[group_table["n_images"] == 1].copy()

# We first place multi-image groups into train by deterministic policy.
# This keeps the known exact duplicate together and avoids artificially
# splitting it across partitions.
assignments = {}

for gid in multi["split_group_id"]:
    assignments[gid] = "train"

# Greedy stratified allocation of singleton groups.
targets = {
    "train": {"benign": int(round((len(df[df["label"] == 0])) * 0.70)),
              "malignant": int(round((len(df[df["label"] == 1])) * 0.70))},
    "validation": {"benign": int(round((len(df[df["label"] == 0])) * 0.15)),
                   "malignant": int(round((len(df[df["label"] == 1])) * 0.15))},
    "test": {"benign": int(round((len(df[df["label"] == 0])) * 0.15)),
             "malignant": int(round((len(df[df["label"] == 1])) * 0.15))},
}

# Shuffle singleton groups reproducibly, then allocate each group to the
# split with the largest remaining weighted need.
single = single.sample(frac=1, random_state=SEED).reset_index(drop=True)

current = {
    s: {"benign": 0, "malignant": 0}
    for s in ["train", "validation", "test"]
}

for _, row in single.iterrows():
    candidates = []

    for split_name in ["train", "validation", "test"]:
        deficit_b = targets[split_name]["benign"] - current[split_name]["benign"]
        deficit_m = targets[split_name]["malignant"] - current[split_name]["malignant"]

        # A group is singleton here, so exactly one class contributes.
        if row["n_benign"]:
            score = deficit_b
        else:
            score = deficit_m

        # Small deterministic preference based on target split fraction.
        fractions = {"train": 0.70, "validation": 0.15, "test": 0.15}
        score = score + 0.0001 * fractions[split_name]
        candidates.append((score, split_name))

    chosen = max(candidates)[1]
    gid = row["split_group_id"]
    assignments[gid] = chosen

    current[chosen]["benign"] += int(row["n_benign"])
    current[chosen]["malignant"] += int(row["n_malignant"])

# Attach split labels.
df["split"] = df["split_group_id"].map(assignments)

if df["split"].isna().any():
    raise AssertionError("Some images were not assigned to a split.")

print("Generated split distribution:")
print(pd.crosstab(df["split"], df["class"]))
print("\nSplit totals:")
print(df["split"].value_counts())


In [ ]:
# Validate the split proportions and leakage control.
print("Proportions:")
print(df["split"].value_counts(normalize=True).sort_index())

# No split group may occur in >1 partition.
group_split_counts = df.groupby("split_group_id")["split"].nunique()
crossing = group_split_counts[group_split_counts > 1]

print("\nGroups crossing partitions:", len(crossing))
assert len(crossing) == 0, "Exact duplicate leakage detected across partitions."

# Every filename occurs exactly once in this cohort.
assert df["filename"].is_unique, "Duplicate filename detected in binary cohort."

# All rows assigned.
assert df["split"].notna().all()

# Reproducibility check: the same seed + source manifest should produce the
# same assignment if this notebook is rerun unchanged.
split_preview = (
    df.groupby(["split", "class"])
      .size()
      .unstack(fill_value=0)
)

display(split_preview)

print("\nPASS: locked split created with no exact-duplicate group crossing partitions.")


In [ ]:
# Save the authoritative Pass 3B split.
split_columns = [
    "image_path",
    "filename",
    "class",
    "filename_number",
    "md5",
    "split_group_id",
    "split",
    "label",
]

split_csv = SPLIT_DIR / "Pass_3B_locked_split.csv"
df[split_columns].sort_values(["split", "class", "filename"]).to_csv(
    split_csv, index=False
)

summary = {
    "seed": SEED,
    "n_binary_images": int(len(df)),
    "n_unique_split_groups": int(df["split_group_id"].nunique()),
    "split_proportions": df["split"].value_counts(normalize=True).to_dict(),
    "class_counts_by_split": {
        split: df[df["split"] == split]["class"].value_counts().to_dict()
        for split in ["train", "validation", "test"]
    },
    "duplicate_groups_crossing_splits": int(len(crossing)),
    "patient_level_split_verified": False,
    "leakage_control_basis": "exact duplicate MD5 grouping",
}

(SPLIT_DIR / "Pass_3B_split_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("LOCKED SPLIT SAVED:")
print(split_csv.resolve())


# 4. Validate representation correspondence

The same locked image membership must be used for all three representation conditions.

The cells below resolve the original image, raw ROI, and preprocessed ROI paths using the configurable project roots.


In [ ]:
def resolve_original(row):
    original = Path(row["image_path"])
    if original.exists():
        return original
    return DATA_ROOT / row["class"] / row["filename"]

def resolve_roi(root, row):
    return root / row["class"] / row["filename"]

df["whole_image_path"] = df.apply(resolve_original, axis=1)
df["raw_roi_path"] = df.apply(lambda r: resolve_roi(RAW_ROI_ROOT, r), axis=1)
df["preprocessed_roi_path"] = df.apply(lambda r: resolve_roi(PREP_ROI_ROOT, r), axis=1)

for col in ["whole_image_path", "raw_roi_path", "preprocessed_roi_path"]:
    missing = df.loc[~df[col].map(lambda x: Path(x).exists()), ["filename", "class", col]]
    print(f"{col}: {len(missing)} missing")
    if len(missing):
        display(missing.head(10))


## Stop here if any representation reports missing files

Do not train with an incomplete cohort.

If all three path checks report zero missing files, continue to the VGG16 baseline cells.


In [ ]:
missing_counts = {
    col: int((~df[col].map(lambda x: Path(x).exists())).sum())
    for col in ["whole_image_path", "raw_roi_path", "preprocessed_roi_path"]
}

if any(v > 0 for v in missing_counts.values()):
    raise FileNotFoundError(
        "One or more required representation datasets are incomplete. "
        f"Missing counts: {missing_counts}"
    )

print("PASS: all three representations are available for every locked binary image.")


# 5. VGG16 baseline

The model is now trained on the locked split.

The validation set is used for early stopping/model selection. The test set remains untouched until final evaluation.

Threshold-dependent metrics use the pre-specified threshold **0.5**. ROC-AUC and PR-AUC are threshold-independent.


In [ ]:
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42

def load_image(path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    return np.asarray(img, dtype=np.float32) / 255.0

def make_arrays(frame, path_col):
    X = np.stack([load_image(p) for p in frame[path_col]])
    y = frame["label"].to_numpy(dtype=np.float32)
    return X, y

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


In [ ]:
def build_vgg16_binary():
    tf.keras.backend.clear_session()

    base = VGG16(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(base.input, output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, roc_curve,
    precision_recall_curve,
)

def run_condition(condition_name, path_col):
    print("\n" + "=" * 80)
    print(condition_name.upper())
    print("=" * 80)

    train_X, train_y = make_arrays(train_df, path_col)
    val_X, val_y = make_arrays(val_df, path_col)
    test_X, test_y = make_arrays(test_df, path_col)

    train_aug = tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.10,
        horizontal_flip=True,
    )
    eval_gen = tf.keras.preprocessing.image.ImageDataGenerator()

    train_gen = train_aug.flow(
        train_X, train_y,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )
    val_gen = eval_gen.flow(
        val_X, val_y,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = build_vgg16_binary()
    model_path = MODEL_DIR / f"vgg16_{condition_name}.keras"

    callbacks = [
        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=7,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
        ),
        ModelCheckpoint(
            model_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
        ),
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    model = tf.keras.models.load_model(model_path)

    prob = model.predict(test_X, batch_size=BATCH_SIZE, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        test_y, pred, labels=[0, 1]
    ).ravel()

    metrics = {
        "condition": condition_name,
        "n_train": len(train_df),
        "n_validation": len(val_df),
        "n_test": len(test_df),
        "accuracy": accuracy_score(test_y, pred),
        "balanced_accuracy": balanced_accuracy_score(test_y, pred),
        "precision": precision_score(test_y, pred, zero_division=0),
        "sensitivity": recall_score(test_y, pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(test_y, pred, zero_division=0),
        "roc_auc": roc_auc_score(test_y, prob),
        "pr_auc": average_precision_score(test_y, prob),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    pred_df = test_df[
        ["image_path", "filename", "class", "label",
         "split", "md5", "split_group_id"]
    ].copy()

    pred_df["condition"] = condition_name
    pred_df["predicted_probability_malignant"] = prob
    pred_df["predicted_label"] = pred
    pred_df["predicted_class"] = np.where(
        pred == 1, "malignant", "benign"
    )

    pred_df.to_csv(
        PRED_DIR / f"{condition_name}_test_predictions.csv",
        index=False,
    )

    pd.DataFrame([metrics]).to_csv(
        OUTPUT_DIR / f"{condition_name}_metrics.csv",
        index=False,
    )

    pd.DataFrame(
        confusion_matrix(test_y, pred, labels=[0, 1]),
        index=["true_benign", "true_malignant"],
        columns=["pred_benign", "pred_malignant"],
    ).to_csv(
        OUTPUT_DIR / f"{condition_name}_confusion_matrix.csv"
    )

    pd.DataFrame(history.history).to_csv(
        OUTPUT_DIR / f"{condition_name}_training_history.csv",
        index=False,
    )

    print(classification_report(
        test_y, pred,
        target_names=["benign", "malignant"],
        digits=4,
        zero_division=0,
    ))

    return metrics, pred_df


In [ ]:
results = []
prediction_frames = []

conditions = [
    ("whole_image", "whole_image_path"),
    ("raw_roi", "raw_roi_path"),
    ("preprocessed_roi", "preprocessed_roi_path"),
]

for name, col in conditions:
    metrics, preds = run_condition(name, col)
    results.append(metrics)
    prediction_frames.append(preds)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "baseline_metrics.csv", index=False)

all_preds = pd.concat(prediction_frames, ignore_index=True)
all_preds.to_csv(OUTPUT_DIR / "all_test_predictions.csv", index=False)

display(results_df.round(4))


In [ ]:
# Final cohort-integrity check
for condition in all_preds["condition"].unique():
    sub = all_preds[all_preds["condition"] == condition]
    assert set(sub["filename"]) == set(test_df["filename"])
    assert sub["split_group_id"].is_unique

assert set(all_preds["condition"]) == {
    "whole_image", "raw_roi", "preprocessed_roi"
}

print("PASS: all three experiments use exactly the same locked test cohort.")
print("PASS: no exact-duplicate group occurs more than once in the test predictions.")


# Pass 3B completion criteria

Pass 3B is complete only when:

- [x] The split is generated from the Pass 3A audit manifest.
- [x] The split is stratified by diagnostic class.
- [x] Exact duplicate groups cannot cross partitions.
- [x] `filename_number` is not treated as patient ID.
- [x] The locked split is saved to disk.
- [x] Whole-image, raw-ROI, and preprocessed-ROI experiments use the same cohort.
- [x] Validation is used for model selection; test data are reserved for final evaluation.
- [x] Test predictions and model checkpoints are saved.

## Important

Do not proceed to Pass 4 based on the historical 95–96% results. Use the **new Pass 3B metrics produced by this locked protocol** as the publication baseline.
